In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

In [6]:
from google.colab import files
uploaded = files.upload()

Saving spam_dataset.csv to spam_dataset (1).csv


In [8]:
import pandas as pd
df = pd.read_csv('spam_dataset.csv').sample(4000, random_state=1).reset_index(drop=True)
df.head()

,text,label
0,"Yep, by the pretty sculpture",ham
1,"Yes, princess. Are you going to make me moan?",ham
2,Welp apparently he retired,ham
3,Havent.,ham
4,I forgot 2 ask ü all smth.. There's a card on ...,ham


In [10]:
import re
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', '', t)
    t = re.sub(r'http\S+', '', t)
    t = re.sub(r'[^a-z\s]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean'] = df['text'].apply(clean_text)
df['len'] = df['clean'].apply(len)
df['exclaim_count'] = df['text'].astype(str).apply(lambda x: x.count('!'))
df['upper_ratio'] = df['text'].astype(str).apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x)+1))

df[['len', 'exclaim_count', 'upper_ratio']].head()

,len,exclaim_count,upper_ratio
0,27,0,0.034483
1,42,0,0.043478
2,26,0,0.037037
3,6,0,0.125000
4,91,0,0.037037


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

X = df[['len', 'exclaim_count', 'upper_ratio']]
y = (df['label'] == 'spam').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

xgb = XGBClassifier()
xgb.fit(X_train, y_train)
pred = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, pred))
print("XGBoost F1 Score:", f1_score(y_test, pred))

XGBoost Accuracy: 0.92875
XGBoost F1 Score: 0.7443946188340808


In [13]:
!pip install transformers -q
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
sample = df['clean'].iloc[0]
tokens = tokenizer(sample, truncation=True, padding='max_length', max_length=64)

print("Original email text:", sample)
print("How BERT sees it as numbers:", tokens['input_ids'][:20])

Original email text: yep by the pretty sculpture
How BERT sees it as numbers: [101, 15624, 2011, 1996, 3492, 6743, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
